<a href="https://colab.research.google.com/github/Raphiyooo/machine-learning/blob/main/uk_weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install seaborn
!pip install sklearn

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [7]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
df = pd.read_csv('land_uk_daily_v2.csv')

In [11]:
df.head()

,date,year,month,day,day_of_year,temp,dewpoint_temp,wind_speed,precipitation,surface_runoff
0,1961-01-01,1961,1,1,1,3.67336,2.07498,4.489435,5.097336,0.145536
1,1961-01-02,1961,1,2,2,3.66230,2.17663,4.753490,10.509815,0.339911
2,1961-01-03,1961,1,3,3,1.80944,0.18316,4.907118,11.650104,0.243362
3,1961-01-04,1961,1,4,4,2.46017,0.17916,4.978218,7.057683,0.148277
4,1961-01-05,1961,1,5,5,1.30547,-0.36810,3.921554,5.822474,0.178550


date caused issues so i dropped it

In [19]:
df.drop('date', axis=1, inplace=True)

now group it by year and month and calculate the mean of every column

In [22]:
df_monthly = df.groupby(['year', 'month']).mean().reset_index()
df_monthly.drop(['day', 'day_of_year'], axis=1, inplace =True)

In [24]:
df_monthly.head()

,year,month,temp,dewpoint_temp,wind_speed,precipitation,surface_runoff
0,1961,1,2.434723,0.622085,4.252304,6.461385,0.213037
1,1961,2,5.693265,3.630586,4.519948,6.277285,0.198630
2,1961,3,7.090844,4.013121,4.298090,3.634951,0.074821
3,1961,4,8.295062,5.397606,3.271359,7.058825,0.123909
4,1961,5,9.715818,5.375655,3.471312,3.541814,0.063128


now it gets interesting, dummy variables are created for the months

one-hot-encoding, converts month into 1s and 0s, drop_first is used to avoid multicollinearity, why drop first, i dont get it exactly, month_1 is missing so if everything else if false month_1 has to be true, why?

In [25]:
final_df = pd.get_dummies(df_monthly, columns=['month'], drop_first=True)

In [29]:
final_df.head(15)

,year,temp,dewpoint_temp,wind_speed,precipitation,surface_runoff,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,1961,2.434723,0.622085,4.252304,6.461385,0.213037,False,False,False,False,False,False,False,False,False,False,False
1,1961,5.693265,3.630586,4.519948,6.277285,0.198630,True,False,False,False,False,False,False,False,False,False,False
2,1961,7.090844,4.013121,4.298090,3.634951,0.074821,False,True,False,False,False,False,False,False,False,False,False
3,1961,8.295062,5.397606,3.271359,7.058825,0.123909,False,False,True,False,False,False,False,False,False,False,False
4,1961,9.715818,5.375655,3.471312,3.541814,0.063128,False,False,False,True,False,False,False,False,False,False,False
5,1961,12.810588,8.235679,3.668521,3.933940,0.037471,False,False,False,False,True,False,False,False,False,False,False
6,1961,13.841611,9.341884,3.597205,5.639185,0.099459,False,False,False,False,False,True,False,False,False,False,False
7,1961,13.827793,9.957879,4.154496,7.632821,0.158559,False,False,False,False,False,False,True,False,False,False,False
8,1961,13.602617,10.168867,3.691854,6.069624,0.131634,False,False,False,False,False,False,False,True,False,False,False
9,1961,9.619891,6.656171,4.665248,9.012595,0.240939,False,False,False,False,False,False,False,False,True,False,False


now create the model

assign X and y

In [32]:
X = final_df.drop(['temp'], axis=1)
y = final_df['temp']

splitting data

In [31]:
from sklearn.model_selection import train_test_split

In [55]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=20)

training the model

In [56]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

In [57]:
print(model.intercept_)
print(model.coef_)

-42.67833326318426
[ 0.02218089  0.60295794 -0.31959186  5.36641304  0.10084926  1.84921157
  4.26531818  7.6925387  10.7097998  12.51668923 12.20652862  9.90427634
  6.57961585  2.95269184  0.68625465]


test the model

In [58]:
y_pred = model.predict(X_test)

In [59]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean absolute error: {mae:.2f}")
print(f"R2: {r2:.2f}")

Mean absolute error: 0.94
R2: 0.92


here you can see dewpoint and temp are closely related

In [61]:
correlation = final_df['temp'].corr(final_df['dewpoint_temp'])
print(f"Correlation between 'temp' and 'dewpoint_temp': {correlation:.2f}")

Correlation between 'temp' and 'dewpoint_temp': 0.99


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='temp', y='dewpoint_temp', data=final_df)
plt.title('Scatter Plot of Temperature vs. Dewpoint Temperature')
plt.xlabel('Temperature')
plt.ylabel('Dewpoint Temperature')
plt.grid(True)
plt.show()

the problem here is as well that the dewpoint_temp is closely related to wind for example because it affects it, so best to just drop it for now

In [52]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd # Ensure pandas is imported

# It's good practice to add a constant to the model for VIF calculation
X_vif = X.copy()
X_vif['intercept'] = 1

# Convert all columns to numeric type to handle boolean columns from get_dummies
X_vif = X_vif.astype(float)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

display(vif_data.sort_values(by='VIF', ascending=False))

,feature,VIF
16,intercept,13528.254462
1,dewpoint_temp,10.558432
10,month_7,8.385738
11,month_8,8.271021
4,surface_runoff,7.042623
9,month_6,6.441010
12,month_9,6.072522
3,precipitation,5.604519
8,month_5,4.146918
13,month_10,3.794345


now drop dewpoint and again calculate vif

In [54]:
X.drop(['dewpoint_temp'], axis=1, inplace=True)

you can see mae rose a lot, by 0.78, r2 score dropped by 0.08

In [60]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean absolute error: {mae:.2f}")
print(f"R2: {r2:.2f}")

Mean absolute error: 0.94
R2: 0.92


In [62]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd # Ensure pandas is imported

# It's good practice to add a constant to the model for VIF calculation
X_vif = X.copy()
X_vif['intercept'] = 1

# Convert all columns to numeric type to handle boolean columns from get_dummies
X_vif = X_vif.astype(float)

vif_data = pd.DataFrame()
vif_data['feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

display(vif_data.sort_values(by='VIF', ascending=False))

,feature,VIF
15,intercept,12416.586824
3,surface_runoff,6.760946
2,precipitation,5.512271
1,wind_speed,2.887435
9,month_7,2.642649
10,month_8,2.574222
8,month_6,2.550554
7,month_5,2.419548
6,month_4,2.208622
11,month_9,2.203622


surface runoff and precipitation have still rather high vif, check the correlation between them

In [63]:
correlation_runoff_precip = X['surface_runoff'].corr(X['precipitation'])
print(f"Correlation between 'surface_runoff' and 'precipitation': {correlation_runoff_precip:.2f}")

Correlation between 'surface_runoff' and 'precipitation': 0.86
